# proof

> other readers' word-level changes to a read document, with who proposed each, reviewed and applied by its owner

In [ ]:
#| default_exp proof

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import tempfile

`Vault.proofread` has each of `readers` (vision models) read a document's pages again and records, per page, the words each reading would change in the current text. A change two readers agree on is `proposed`; one only one reader makes is a `doubt`. The owner reviews them: `Vault.changes` lists them, `Vault.accept` applies the chosen ones as a new version of each page (route `proof`) and re-indexes the document, `Vault.reject` sets them aside, and `proof_md` writes the record of what changed and what was not taken.

In [ ]:
#| export
import json, re, time, uuid
from difflib import SequenceMatcher
from fastcore.all import AttrDict, L, Path, first, ifnone, patch
from litesearch import write_txn
from vishalakshi.core import Vault
from vishalakshi.read import read_page, render, as_list, IMG_EXTS

## Word changes

`word_changes` compares two texts word by word and returns each replaced, inserted or deleted run as `{at, n, before, after}`: `at` is the first word's index in `a`, `n` how many of `a`'s words it replaces. `apply_changes` makes them, last first, keeping the text's own spacing elsewhere.

In [ ]:
#| export
_TOK = re.compile(r'\S+|\s+')
def _words(t): return [w for w in _TOK.findall(t or '') if not w.isspace()]

def word_changes(a:str, b:str) -> list:
    "The word runs `b` changes in `a`, as `{at, n, before, after}`."
    wa, wb = _words(a), _words(b)
    return [dict(at=i1, n=i2 - i1, before=' '.join(wa[i1:i2]), after=' '.join(wb[j1:j2]))
            for op, i1, i2, j1, j2 in SequenceMatcher(None, wa, wb, autojunk=False).get_opcodes() if op != 'equal']

def apply_changes(t:str, changes) -> str:
    "`t` with `changes` made: each replaces `n` words from word `at` by `after`; the text's own spacing kept elsewhere."
    toks = _TOK.findall(t or '')
    lead, out = (toks[0] if toks and toks[0].isspace() else ''), []
    for x in toks:
        if not x.isspace(): out.append([x, ''])
        elif out: out[-1][1] = x
    for c in sorted(changes, key=lambda c: -c['at']):
        a, e = c['at'], c['at'] + c['n']
        new = [[w, ' '] for w in (c['after'] or '').split()]
        if new and c['n']: new[-1][1] = out[e - 1][1]
        elif new and a >= len(out):
            new[-1][1] = out[-1][1] if out else ''
            if out: out[-1][1] = out[-1][1] or ' '
        elif not new and c['n'] and e == len(out) and a: out[a - 1][1] = out[e - 1][1]
        out[a:e] = new
    return lead + ''.join(w + s for w, s in out)

In [ ]:
_a, _b = 'धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवः', 'धर्मक्षेत्रे कुरुक्षेत्रे समवेता युयुत्सवाः ॥१॥'
_c = word_changes(_a, _b)
test_eq(_c, [dict(at=3, n=1, before='युयुत्सवः', after='युयुत्सवाः ॥१॥')])
test_eq(apply_changes(_a, _c), _b)
test_eq(apply_changes('a b c', word_changes('a b c', 'a c')), 'a c')
test_eq(apply_changes('a c', word_changes('a c', 'a b c')), 'a b c')
test_eq(apply_changes('line one\nline two', word_changes('line one\nline two', 'line 1\nline two')), 'line 1\nline two')
for x, y in [('', 'new words'), ('a', ''), ('x y z', 'z y x'), ('one two', 'one two three')]: test_eq(apply_changes(x, word_changes(x, y)), y if x or y else '')

## Proofreading

`readers` are model names; `chats` maps a reader to a stand-in callable, as the tests do. `agree` readers must make the same change for it to be `proposed`. Re-running replaces a page's open changes, so a proofread is repeatable.

In [ ]:
#| export
@patch
def _proof_t(self:Vault):
    t = self.db.t.proof_changes
    if not t.exists():
        with write_txn(self.db):
            t.create(id=str, doc_id=str, store=str, page=int, version=int, at=int, n=int, before=str, after=str, by=str, state=str, ts=float, pk='id')
    return t

@patch
def proofread(self:Vault, doc_id:str, readers, pages=None, agree:int=2, preset:str=None, script:str=None, hint:str=None,
              dpi:int=200, chats:dict=None,
              pii:str='redact',    # what `read_page` applies: a reader that may not see a page is skipped for it and reported in `blocked`
              allow:bool=False,    # allow once, for Personal pages
) -> dict:
    "Each of `readers` reads `pages` (all when None) again; their word changes to the current text are filed, `proposed` where `agree` agree."
    d, readers, pages = first(self.t.docs(where='id=?', where_args=[doc_id])), as_list(readers), as_list(pages, int)
    assert d, f'no document {doc_id}'
    m, src, t = json.loads(d['meta'] or '{}'), Path(d['source']), self._proof_t()
    preset, script, hint = ifnone(preset, m.get('preset', 'generic')), ifnone(script, m.get('script', '')), ifnone(hint, m.get('hint', ''))
    cur = [p for p in self.pages(doc_id) if pages is None or p.page in set(pages)]
    n, blocked = 0, []
    for p in cur:
        img = src.read_bytes() if src.suffix.lower() in IMG_EXTS else render(src, p.page, dpi)
        got, lv, read = {}, self.level(doc_id, page=p.page).level, 0
        for r in readers:
            ch = (chats or {}).get(r)
            pg = read_page(img, r, preset, d['title'], script, hint, chat=ch, pii=pii, level=lv, text=p.text, allow=allow)
            if pg.get('blocked'):
                blocked.append(dict(page=p.page, reader=r, **pg.blocked)); continue
            read += 1
            self._read_sent(doc_id, [AttrDict(page=p.page, route='model', model=getattr(ch, 'model', r))], r, ch, pii)
            for c in word_changes(p.text, pg.text): got.setdefault((c['at'], c['n'], c['after']), dict(c, by=[]))['by'].append(r)
        if not read: continue
        with write_txn(self.db):
            self.db.execute('delete from proof_changes where doc_id=? and store=? and page=? and state in (\'proposed\', \'doubt\')', [doc_id, self.name, p.page])
            for c in got.values():
                t.insert(dict(id=uuid.uuid4().hex[:12], doc_id=doc_id, store=self.name, page=p.page, version=p.version, at=c['at'], n=c['n'],
                              before=c['before'], after=c['after'], by=json.dumps(c['by']), state='proposed' if len(c['by']) >= agree else 'doubt', ts=time.time()))
                n += 1
    return dict(doc_id=doc_id, pages=[p.page for p in cur], changes=n, proposed=len(self.changes(doc_id, 'proposed')), blocked=blocked)

@patch
def changes(self:Vault, doc_id:str, state:str=None, page:int=None) -> L:
    "A document's filed changes, by page and position: `proposed`, `doubt`, `accepted` or `rejected`."
    w, a = 'doc_id=? and store=?', [doc_id, self.name]
    if state: w, a = w + ' and state=?', a + [state]
    if page is not None: w, a = w + ' and page=?', a + [page]
    return L(AttrDict(r, by=json.loads(r['by'])) for r in self._proof_t().rows_where(w, a, order_by='page, at'))

@patch
def reject(self:Vault, doc_id:str, ids) -> int:
    "Set changes aside; they stay in the record as not taken."
    ids = as_list(ids)
    with write_txn(self.db):
        for i in ids: self._proof_t().update(dict(id=i, state='rejected'))
    return len(ids)

@patch
def accept(self:Vault, doc_id:str, ids=None) -> dict:
    "Apply changes, the `proposed` ones when `ids` is None, as a new version of each page they touch, and re-index the document."
    ids = as_list(ids)
    cs = self.changes(doc_id).filter(lambda c: c.id in set(ids) if ids is not None else c.state == 'proposed')
    pg = {p.page: p for p in self.pages(doc_id)}
    stale = [c.id for c in cs if pg.get(c.page) is None or pg[c.page].version != c.version]
    assert not stale, f'changes {stale} were made against an older version of their page: proofread again'
    rows = L()
    for k, g in cs.groupby(lambda c: c.page).items():
        rows.append(AttrDict(page=k, text=apply_changes(pg[k].text, g), route='proof', flags=pg[k].flags, model=None))
    if rows:
        from vishalakshi.read import flags
        for r in rows: r.flags = flags(r.text)
        self._file_pages(doc_id, rows, None)
        with write_txn(self.db):
            for c in cs: self._proof_t().update(dict(id=c.id, state='accepted'))
        d = first(self.t.docs(where='id=?', where_args=[doc_id]))
        new = {p.page: p for p in self.pages(doc_id)}
        self.add([(k, new[k].text) for k in sorted(new)], d['title'], source=d['source'], kind=d['kind'], meta=json.loads(d['meta'] or '{}'), force=True)
    return dict(doc_id=doc_id, accepted=len(cs), pages=[r.page for r in rows])

@patch
def proof_md(self:Vault, doc_id:str) -> str:
    "The record of a document's proofreading: what changed and what was not taken, by page."
    d = first(self.t.docs(where='id=?', where_args=[doc_id]))
    line = lambda c: f"- p. {c.page + 1}: {c.before or '∅'} → {c.after or '∅'} ({', '.join(c.by)})"
    out = [f"# Proofread: {d['title'] if d else doc_id}"]
    for title, st in (('Changed', 'accepted'), ('Proposed', 'proposed'), ('In doubt', 'doubt'), ('Not taken', 'rejected')):
        if (cs := self.changes(doc_id, st)): out += ['', f'## {title}', '', *map(line, cs)]
    return '\n'.join(out) + '\n'

In [ ]:
from vishalakshi.read import read_doc
from pdflite import pdf_doc
class Says:
    "A reader that reads every page as `text`."
    def __init__(self, text): self.text, self.model = text, 'fake'
    def __call__(self, msg): return dict(content=json.dumps(dict(items=[dict(kind='prose', text=self.text)])))
_d = Path(tempfile.mkdtemp())
from pdf_oxide import Pdf
_fx = Path(tempfile.mkdtemp())
for _i, _t in enumerate(['# Attention Is All You Need\n\nThe dominant sequence transduction models are based on recurrent or convolutional networks.',
                         '# Background\n\nThe goal of reducing sequential computation also forms the foundation of the Extended Neural GPU.']):
    Pdf.from_markdown(_t).save(str(_fx/f'p{_i}.pdf'))
Pdf.merge([str(_fx/'p0.pdf'), str(_fx/'p1.pdf')]).save(str(_fx/'two.pdf'))
_src = _fx/'two.pdf'
_scan = _d/'scan.pdf'; _scan.write_bytes(pdf_doc(pdf_doc(_src).extract_pages_to_bytes([0])).flatten_to_images(dpi=72))
_v = Vault(Path(tempfile.mkdtemp())/'v.db', offline=True)
_o = _v.read_doc(_scan, parse='llm', chat=Says('the dominat sequence transduction models'))
_id = _o['doc_id']
_r = _v.proofread(_id, ['a', 'b', 'c'], chats=dict(a=Says('the dominant sequence transduction models'), b=Says('the dominant sequence transduction models'),
                                                   c=Says('the dominat sequence transduktion models')))
test_eq(_r['changes'], 2); test_eq(_r['proposed'], 1)
_p = _v.changes(_id, 'proposed')[0]; test_eq((_p.before, _p.after, _p.by), ('dominat', 'dominant', ['a', 'b']))
_dt = _v.changes(_id, 'doubt')[0]; test_eq((_dt.after, _dt.by), ('transduktion', ['c']))
_v.reject(_id, [_dt.id])
test_eq(_v.accept(_id)['accepted'], 1)
test_eq(_v.pages(_id)[0].text, 'the dominant sequence transduction models'); test_eq(_v.pages(_id)[0].route, 'proof')
assert 'dominant' in _v.search('dominant sequence')[0]['snippet']
_md_ = _v.proof_md(_id)
assert '## Changed' in _md_ and 'dominat → dominant (a, b)' in _md_ and '## Not taken' in _md_
_v.proofread(_id, ['a'], chats=dict(a=Says('the dominant sequence transduction model')))
_v.reparse(_id, chat=Says('a different reading entirely'))
test_fail(lambda: _v.accept(_id, [c.id for c in _v.changes(_id, 'doubt')]), contains='older version')

In [ ]:
#| hide
# a hosted reader may not see a Personal page: it is skipped and reported, and the page's proposals stand
_pv = Vault(Path(tempfile.mkdtemp())/'pv.db', offline=True)
_po = _pv.read_doc(_scan, parse='llm', chat=Says('write to ada@example.com about the dominat models'))
_pid = _po['doc_id']
_r = _pv.proofread(_pid, ['a'], chats=dict(a=Says('write to ada@example.com about the dominant models')))
test_eq((_r['changes'], [(b['page'], b['reader'], b['kind']) for b in _r['blocked']]), (0, [(0, 'a', 'pii_blocked')]))
# allowed once, it reads; a local reader needs no allowing
_r = _pv.proofread(_pid, ['a'], chats=dict(a=Says('write to ada@example.com about the dominant models')), allow=True)
test_eq((_r['changes'], _r['blocked']), (1, []))
class LocalSays(Says): runtime = 'litert'
_r = _pv.proofread(_pid, ['b'], chats=dict(b=LocalSays('write to ada@example.com about the dominant models')))
test_eq(_r['blocked'], [])
# Local only stops even an allowed hosted reader
_pv.set_level(_pid, 'local')
test_eq(_pv.proofread(_pid, ['a'], chats=dict(a=Says('x')), allow=True)['blocked'][0]['kind'], 'local_only')


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()